# 05 | Burkina Faso: preparing crop reference data
**Ibrahim Kabore · WorldCereal extension · Koumbia, Burkina Faso**

The French practical started with ready-made satellite embeddings. Here, I begin with real agricultural field observations and prepare a defensible reference dataset before extracting satellite features.

The goal is to classify six crops using historical labels and to evaluate performance in unseen years and locations. This notebook investigates the JECAM field polygons; the next notebook prepares the independently available WorldCereal satellite extractions. The two datasets are not assumed to share identical sampling points.

**Data:** [JECAM-CIRAD paper](https://doi.org/10.5194/essd-13-5951-2021) · [GeoParquet distribution](https://source.coop/fiboa/data/jecam) (CC BY 4.0). Run in Colab using the CPU.

## 1. Set up the project
Keep the raw data and generated files on Drive so they survive runtime resets.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import requests

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/ESA_EO_Projects/WorldCereal_Burkina_Faso")
DATA = PROJECT / "data"
for folder in ["data", "notebooks", "models", "outputs"]:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)
print(PROJECT)

## 2. Download and inspect JECAM
The GeoParquet contains geographically referenced field polygons from several countries. Select Burkina Faso using its ISO country code. An observation is a field record for a particular date, not necessarily a unique physical field.

In [ ]:
URL = "https://data.source.coop/fiboa/data/jecam/jecam.parquet"
path = DATA / "jecam.parquet"
if not path.exists():
    response = requests.get(URL, timeout=120)
    response.raise_for_status()
    assert response.content[:4] == b"PAR1"
    path.write_bytes(response.content)

gdf = gpd.read_parquet(path)
burkina = gdf.loc[gdf["admin:country_code"].eq("BF")].copy()
burkina["year"] = pd.to_datetime(burkina["determination:datetime"]).dt.year
print("Burkina Faso observations:", len(burkina))
display(burkina["crop:name"].value_counts(dropna=False))
display(burkina["year"].value_counts().sort_index())
display(burkina["site_name"].value_counts())

## 3. Choose years and crops
Use 2017 onward for the historical Sentinel workflow. Separate years rather than randomly mixing records: 2017, 2018 and 2020 for training, 2021 for validation and 2022 for testing. Choose the six well-represented pure crop labels; do not treat missing labels, fallow or mixed crops as known crop types.

In [ ]:
SPLIT = {2017: "train", 2018: "train", 2020: "train", 2021: "validation", 2022: "test"}
CLASSES = ["Maize", "Cotton", "Soybean", "Groundnut", "Cowpea", "Sorghum"]
print("Missing crop names:", burkina["crop:name"].isna().sum())
recent = burkina.loc[burkina.year.isin(SPLIT)].copy()
display(pd.crosstab(recent["crop:name"].fillna("Missing"), recent.year, margins=True))
reference = recent.loc[recent["crop:name"].isin(CLASSES)].copy().reset_index(drop=True)
reference["label"] = reference["crop:name"]
reference["split"] = reference.year.map(SPLIT)
reference["sample_id"] = [f"BF_KOUMBIA_{y}_{i:05d}" for i,y in enumerate(reference.year)]
print("Selected:", len(reference))
display(pd.crosstab(reference.label, reference.split, margins=True))

In [ ]:
ax = reference.plot(column="label", legend=True, cmap="tab10", figsize=(12,8),
                    legend_kwds={"bbox_to_anchor": (1.02,1)})
ax.set(title="JECAM reference fields | Koumbia", xlabel="Longitude", ylabel="Latitude")
plt.tight_layout()
plt.show()
print("Extent:", reference.total_bounds)

## 4. Field sizes and interior sampling
Reproject to UTM 30N to measure metres and hectares. A 10 m pixel covers 0.01 ha, but some Sentinel-2 bands are natively 20 m. A 10 m inward buffer reduces edge contamination; it does not guarantee a pure pixel. Inspect the class-specific losses because smaller fields are more likely to disappear.

In [ ]:
reference_utm = reference.to_crs("EPSG:32630")
assert reference_utm.geometry.notna().all()
assert reference_utm.geometry.is_valid.all() and not reference_utm.geometry.is_empty.any()
reference_utm["area_ha"] = reference_utm.geometry.area / 10000
display(reference_utm.area_ha.describe())
display(reference_utm.groupby("label").area_ha.median())

buffered = reference_utm.copy()
buffered["geometry"] = buffered.geometry.buffer(-10)
usable = buffered.loc[~buffered.geometry.is_empty].copy()
print("Original:", len(reference_utm), "Retained:", len(usable), "Excluded:", len(reference_utm)-len(usable))
retention = pd.concat([reference_utm.label.value_counts().rename("original"),
                       usable.label.value_counts().rename("retained")], axis=1)
retention["retention_pct"] = 100 * retention.retained / retention.original
display(retention.round(2))
display(pd.crosstab(usable.label, usable.split, margins=True))

In [ ]:
samples = usable.copy()
samples["geometry"] = samples.geometry.representative_point()
samples = samples.to_crs("EPSG:4326")
samples["lon"], samples["lat"] = samples.geometry.x, samples.geometry.y
assert samples.sample_id.is_unique
display(samples[["sample_id","label","year","split","lon","lat"]].head())
ax = samples.plot(column="label", legend=True, cmap="tab10", markersize=5, figsize=(12,8))
ax.set(title="Interior sampling points | Koumbia", xlabel="Longitude", ylabel="Latitude")
plt.show()

## 5. Diagnose spatial dependence
A chronological split tests transfer across years, but the same fields or nearby fields can appear in different years. Use nearest-neighbour distances to measure this dependence. A short distance is evidence of proximity, not proof of parcel identity.

In [ ]:
from scipy.spatial import cKDTree
spatial = samples.to_crs("EPSG:32630").copy()
xy = np.column_stack([spatial.geometry.x, spatial.geometry.y])
tree = cKDTree(xy[spatial.split.eq("train")])
for part in ["validation","test"]:
    distances, _ = tree.query(xy[spatial.split.eq(part)])
    print(part, "| median distance (m):", round(np.median(distances),1))
    for threshold in [10,50,100,500]:
        print(f"  Within {threshold} m: {(distances <= threshold).mean()*100:.1f}%")

## 6. Define spatial groups
Assign every point to a 2 km square in the projected coordinate grid. Later, use group-aware cross-validation to withhold whole blocks; adjacent blocks can still share spatial dependence. Colours below illustrate groups, not crop labels.

In [ ]:
BLOCK_SIZE = 2000
spatial["spatial_group"] = (
    (spatial.geometry.x // BLOCK_SIZE).astype(int).astype(str) + "_" +
    (spatial.geometry.y // BLOCK_SIZE).astype(int).astype(str)
)
samples["spatial_group"] = spatial["spatial_group"].values
print("Occupied blocks:", spatial.spatial_group.nunique())
display(spatial.groupby("spatial_group").size().describe())
display(pd.crosstab(spatial.spatial_group, spatial.label))
ax = spatial.plot(column="spatial_group", cmap="turbo", categorical=True,
                  markersize=5, figsize=(12,8))
ax.set(title="Spatial validation blocks | Koumbia", xlabel="Easting (m)", ylabel="Northing (m)")
plt.show()

## 7. Save the reference products
Save field polygons, sampled points and split assignments. These points are useful for future custom extractions, but are not the identifiers of the already-extracted WorldCereal samples.

In [ ]:
samples.to_parquet(DATA / "koumbia_reference_points.parquet", index=False)
samples.to_file(DATA / "koumbia_reference_points.geojson", driver="GeoJSON", index=False)
reference_utm.to_file(DATA / "koumbia_reference_fields.gpkg", driver="GPKG", index=False)
print("Reference datasets saved to", DATA)

## 8. Results from the original run
The original exploration found 7,985 Burkina Faso records, 4,808 in the selected years and 3,150 records from the six crop classes. Buffering retained 2,964 observations (94.1%); 50 spatial blocks. Median distance to training was 13.0 m for validation and 16.5 m for test. Nearly 90% of test points were within 50 m of training. These are recorded results, not freshly executed outputs. Notebook 06 discovers existing WorldCereal extractions, which require their own sample counts and spatial groups.